# GS-Verse Asset Pipeline: GaMeS Training

Train GaMeS on a NeRF-synthetic dataset (from BlenderNeRF) and export assets for Unity.

**Prerequisites:** You already have a dataset zip from BlenderNeRF (`train/` images + `transforms_train.json` + mesh `.obj` file).

**Runtime:** Select GPU: `Runtime > Change runtime type > T4 GPU`

## Step 1: Check GPU & Environment

In [ ]:
# Verify GPU is available
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("ERROR: No GPU! Go to Runtime > Change runtime type > T4 GPU")

## Step 2: Upload dataset + mesh

Upload **two files**:
1. Your BlenderNeRF dataset zip (contains `train/` folder with images + `transforms_train.json`)
2. Your mesh `.obj` file (exported from Blender or converted from `.glb`)

If you only have a `.glb` file, the notebook will convert it to `.obj` for you.

In [ ]:
from google.colab import files
import os, zipfile, shutil, glob, json

WORK_DIR = "/content/gsverse_pipeline"
DATASET_DIR = os.path.join(WORK_DIR, "mesh_data")

# Clean previous runs
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(DATASET_DIR, exist_ok=True)

# --- Upload dataset zip ---
print("=== Upload your dataset zip ===")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(DATASET_DIR)
os.remove(zip_name)

# --- Fix transforms JSON paths for GaMeS ---
# GaMeS expects: "./train/0001" (with ./ prefix, no .png extension)
# BlenderNeRF may output: "train\\0001.png" or "train/0001.png" or "train/0001"
for tf_name in ["transforms_train.json", "transforms_test.json", "transforms_val.json"]:
    tf_path = os.path.join(DATASET_DIR, tf_name)
    if not os.path.exists(tf_path):
        continue
    with open(tf_path, "r") as f:
        data = json.load(f)
    changed = False
    for frame in data.get("frames", []):
        fp = frame.get("file_path", "")
        new_fp = fp.replace("\\", "/")
        if new_fp.endswith(".png"):
            new_fp = new_fp[:-4]
        if not new_fp.startswith("./"):
            new_fp = "./" + new_fp
        if new_fp != fp:
            frame["file_path"] = new_fp
            changed = True
    if changed:
        with open(tf_path, "w") as f:
            json.dump(data, f, indent=2)
        print(f"Fixed paths in {tf_name}")
        if data.get("frames"):
            print(f"  Example: {data['frames'][0]['file_path']}")
    else:
        print(f"{tf_name} paths OK")

# Count images
img_files = []
for ext in ["*.png", "*.jpg"]:
    img_files.extend(glob.glob(os.path.join(DATASET_DIR, "**", ext), recursive=True))
print(f"\nFound {len(img_files)} images")

# Show dataset structure
print("\n=== Dataset structure ===")
for root, dirs, fls in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in sorted(fls)[:5]:
            print(f"{indent}  {f}")
        if len(fls) > 5:
            print(f"{indent}  ... and {len(fls) - 5} more files")

In [ ]:
# --- Upload mesh file (.obj or .glb) ---
print("\n=== Upload your mesh file (.obj or .glb) ===")
uploaded_mesh = files.upload()
mesh_name = list(uploaded_mesh.keys())[0]

mesh_dest = os.path.join(DATASET_DIR, "mesh.obj")

if mesh_name.endswith(".glb") or mesh_name.endswith(".gltf"):
    # Convert GLB -> OBJ using trimesh
    !pip install -q trimesh
    import trimesh
    scene = trimesh.load(mesh_name)
    if isinstance(scene, trimesh.Scene):
        mesh = trimesh.util.concatenate(scene.dump())
    else:
        mesh = scene
    mesh.export(mesh_dest)
    os.remove(mesh_name)
    print(f"Converted {mesh_name} -> mesh.obj")
elif mesh_name.endswith(".obj"):
    shutil.move(mesh_name, mesh_dest)
    print(f"Moved {mesh_name} -> mesh.obj")
else:
    print(f"ERROR: Unsupported format: {mesh_name}. Use .obj or .glb")

print(f"Mesh saved to: {mesh_dest}")
print(f"Mesh size: {os.path.getsize(mesh_dest) / 1024:.0f} KB")

# Also copy for Unity export later
shutil.copy(mesh_dest, os.path.join(WORK_DIR, "mesh_final.obj"))

# Remove sparse/ if it exists (would trigger Colmap loader instead of Blender)
sparse_dir = os.path.join(DATASET_DIR, "sparse")
if os.path.exists(sparse_dir):
    shutil.rmtree(sparse_dir)
    print("Removed sparse/ dir (prevents Colmap loader conflict)")

print("\n=== Dataset ready ===")
for root, dirs, fls in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in sorted(fls)[:5]:
            print(f"{indent}  {f}")
        if len(fls) > 5:
            print(f"{indent}  ... and {len(fls) - 5} more files")

## Step 3: Install GaMeS

This clones the GaMeS repo and builds CUDA extensions. If the build fails, try:
- `Runtime > Restart runtime` and re-run from this cell
- Or switch to a different GPU type in `Runtime > Change runtime type`

In [ ]:
import os, torch, subprocess

print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

# --- Setup CUDA environment ---
nvcc = subprocess.run(["which", "nvcc"], capture_output=True, text=True)
if nvcc.returncode != 0:
    cuda_dirs = sorted([d for d in os.listdir("/usr/local") if d.startswith("cuda-")])
    if cuda_dirs:
        cuda_home = f"/usr/local/{cuda_dirs[-1]}"
    elif os.path.exists("/usr/local/cuda"):
        cuda_home = "/usr/local/cuda"
    else:
        raise RuntimeError("CUDA not found! Make sure you're using a GPU runtime.")
    os.environ["CUDA_HOME"] = cuda_home
    os.environ["PATH"] = f"{cuda_home}/bin:" + os.environ["PATH"]
    print(f"Set CUDA_HOME={cuda_home}")
else:
    cuda_home = os.path.dirname(os.path.dirname(nvcc.stdout.strip()))
    os.environ["CUDA_HOME"] = cuda_home
    print(f"CUDA_HOME={cuda_home}")

!nvcc --version | tail -1

# --- Install build tools ---
!pip install -q ninja

os.chdir("/content")

# --- Clone GaMeS ---
if not os.path.exists("gaussian-mesh-splatting"):
    !git clone https://github.com/waczjoan/gaussian-mesh-splatting.git

os.chdir("gaussian-mesh-splatting")
!git submodule update --init --recursive

# --- Patch simple-knn: fix missing FLT_MAX in CUDA 12+ ---
knn_file = "submodules/simple-knn/simple_knn.cu"
with open(knn_file, "r") as f:
    code = f.read()
if "#include <cfloat>" not in code and "#include <float.h>" not in code:
    code = "#include <cfloat>\n" + code
    with open(knn_file, "w") as f:
        f.write(code)
    print("Patched simple_knn.cu: added #include <cfloat>")

# --- Install Python deps (including smplx for FLAME module) ---
!pip install -q plyfile trimesh scipy tqdm smplx

# --- Build CUDA extensions ---
print("\n--- Building diff-gaussian-rasterization (~2 min) ---")
!pip install -v --no-build-isolation submodules/diff-gaussian-rasterization 2>&1 | tail -3

print("\n--- Building simple-knn ---")
!pip install -v --no-build-isolation submodules/simple-knn 2>&1 | tail -3

# --- Verify ---
ok = True
for mod in ["diff_gaussian_rasterization", "simple_knn"]:
    try:
        __import__(mod)
        print(f"{mod} OK")
    except ImportError as e:
        print(f"{mod} FAILED: {e}")
        ok = False

if ok:
    print("\nAll good! Proceed to Step 4.")
else:
    print("\nSomething failed. Try: Runtime > Restart runtime, then re-run this cell.")

## Step 4: Train GaMeS

This trains Gaussian Mesh Splatting on your dataset. Takes ~5-15 min on T4 GPU.

In [ ]:
os.chdir("/content/gaussian-mesh-splatting")

DATASET_DIR = "/content/gsverse_pipeline/mesh_data"
OUTPUT_DIR = "/content/gsverse_pipeline/output/mesh_5splats_per_face"

!python train.py --eval \
    -s "{DATASET_DIR}" \
    -m "{OUTPUT_DIR}" \
    --gs_type gs_mesh \
    --num_splats 5 \
    -w

print("\nTraining complete!")
print("Output files:")
for root, dirs, fls in os.walk(OUTPUT_DIR):
    for f in fls:
        fp = os.path.join(root, f)
        print(f"  {os.path.relpath(fp, OUTPUT_DIR)} ({os.path.getsize(fp)/1024:.0f} KB)")

## Step 5: Export for Unity

In [ ]:
import torch
import json

WORK_DIR = "/content/gsverse_pipeline"
OUTPUT_DIR = "/content/gsverse_pipeline/output/mesh_5splats_per_face"

# --- Find model_params.pt ---
params_path = None
for root, dirs, fls in os.walk(OUTPUT_DIR):
    for f in fls:
        if f == "model_params.pt":
            params_path = os.path.join(root, f)
            break
    if params_path:
        break

if not params_path:
    raise FileNotFoundError("model_params.pt not found in output! Check training logs above.")

print(f"Loading: {params_path}")
model_weights = torch.load(params_path, map_location="cpu", weights_only=False)

# Extract alpha and scale
alpha = model_weights.get("_alpha")
scale = model_weights.get("_scale")

if alpha is None or scale is None:
    print(f"Available keys: {list(model_weights.keys())}")
    raise KeyError("_alpha or _scale not found in model_params.pt")

# Handle nested tensors
if isinstance(alpha, (list, tuple)) and len(alpha) == 1:
    alpha = alpha[0]
if isinstance(scale, (list, tuple)) and len(scale) == 1:
    scale = scale[0]

alpha_np = alpha.detach().cpu().numpy()
scale_np = scale.detach().cpu().numpy()

model_data = {
    "_alpha": alpha_np.tolist(),
    "_scale": scale_np.tolist()
}

output_json = os.path.join(WORK_DIR, "model_params.json")
with open(output_json, "w") as f:
    json.dump(model_data, f)

print(f"Saved: model_params.json")
print(f"Alpha shape: {alpha_np.shape}")
print(f"Scale shape: {scale_np.shape}")

# --- Find and copy point_cloud.ply ---
ply_path = None
for root, dirs, fls in os.walk(OUTPUT_DIR):
    for f in fls:
        if f == "point_cloud.ply":
            ply_path = os.path.join(root, f)
            break
    if ply_path:
        break

if not ply_path:
    raise FileNotFoundError("point_cloud.ply not found! Check training logs.")

shutil.copy(ply_path, os.path.join(WORK_DIR, "point_cloud.ply"))
print(f"Copied: point_cloud.ply ({os.path.getsize(ply_path) / 1024:.0f} KB)")

print("\n=== All files ready ===")
for f in ["point_cloud.ply", "model_params.json", "mesh_final.obj"]:
    p = os.path.join(WORK_DIR, f)
    if os.path.exists(p):
        print(f"  {f} ({os.path.getsize(p) / 1024:.0f} KB)")
    else:
        print(f"  {f} - MISSING")

## Step 6: Download results

In [ ]:
from google.colab import files
import zipfile

WORK_DIR = "/content/gsverse_pipeline"
zip_path = os.path.join(WORK_DIR, "unity_asset_key.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["point_cloud.ply", "model_params.json", "mesh_final.obj"]:
        p = os.path.join(WORK_DIR, f)
        if os.path.exists(p):
            zf.write(p, f)
            print(f"Added: {f}")
        else:
            print(f"SKIPPED (missing): {f}")

print(f"\nZip: {os.path.getsize(zip_path) / 1024:.0f} KB")
print("Downloading...")
files.download(zip_path)

## What to do next in Unity

1. Unzip `unity_asset_key.zip`
2. Copy `mesh_final.obj` -> `Assets/Resources/key.obj`
3. Copy `point_cloud.ply` and `model_params.json` -> `Assets/RoomScenes/darkRoom/key/`
4. In Unity: **Tools > Gaussian Splats > Create GaussianSplatAsset**
   - **Processing Mode** = **GaMeS**
   - **L-Handed Coordinate System** = checked
   - **Input point cloud** = `point_cloud.ply`
   - **Input json params** = `model_params.json`
   - **Path to obj** = `key` (name in Resources, no extension)
5. Choose output folder and quality
6. Click **Create Asset**
7. Drag the created asset into your scene